In [5]:
# CELL 1 — Imports & Load
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('/kaggle/input/competitions/ai-legends-2026-model-development/train.csv', low_memory=False)
train['SaleDate']  = pd.to_datetime(train['SaleDate'],  errors='coerce')
train['WeekStart'] = pd.to_datetime(train['WeekStart'], errors='coerce')
print("train:", train.shape)
print(train['TaskType'].value_counts())

train: (367120, 7)
TaskType
product_weekly    352039
category_daily      8226
store_daily         6855
Name: count, dtype: int64


In [6]:
# CELL 2 — Sparse байдлыг шинжлэх (product_weekly)
# Ямар хувь нь бага зарагддаг болохыг харна
pw = train[train['TaskType'] == 'product_weekly'].copy()

# Бүтээгдэхүүн тус бүрийн дундаж долоо хоногийн борлуулалт
prod_avg = pw.groupby('Product')['SalesQty'].mean()

print("=== Product Weekly Sparsity Analysis ===")
print(f"Нийт бүтээгдэхүүн:          {prod_avg.shape[0]}")
print(f"Дундаж < 1  (маш бага):     {(prod_avg < 1).sum()}  ({(prod_avg < 1).mean()*100:.1f}%)")
print(f"Дундаж < 5  (бага):         {(prod_avg < 5).sum()}  ({(prod_avg < 5).mean()*100:.1f}%)")
print(f"Дундаж >= 5 (хэвийн):       {(prod_avg >= 5).sum()}  ({(prod_avg >= 5).mean()*100:.1f}%)")
print()
print("SalesQty тархалт:")
print(pw['SalesQty'].describe())
print(f"\nТэгтэй мөр: {(pw['SalesQty']==0).sum()} ({(pw['SalesQty']==0).mean()*100:.1f}%)")

=== Product Weekly Sparsity Analysis ===
Нийт бүтээгдэхүүн:          1787
Дундаж < 1  (маш бага):     496  (27.8%)
Дундаж < 5  (бага):         1246  (69.7%)
Дундаж >= 5 (хэвийн):       541  (30.3%)

SalesQty тархалт:
count    352039.000000
mean          6.234701
std          22.376874
min           0.000000
25%           0.000000
50%           0.000000
75%           5.000000
max        1718.000000
Name: SalesQty, dtype: float64

Тэгтэй мөр: 199631 (56.7%)


In [7]:
# CELL 3 — Helpers
def add_date_features(df, date_col):
    df = df.copy()
    d = df[date_col]
    df['feat_year']       = d.dt.year
    df['feat_month']      = d.dt.month
    df['feat_day']        = d.dt.day
    df['feat_dayofweek']  = d.dt.dayofweek
    df['feat_dayofyear']  = d.dt.dayofyear
    df['feat_weekofyear'] = d.dt.isocalendar().week.astype(int)
    df['feat_quarter']    = d.dt.quarter
    df['feat_is_weekend'] = (d.dt.dayofweek >= 5).astype(int)
    df['feat_month_sin']  = np.sin(2 * np.pi * d.dt.month / 12)
    df['feat_month_cos']  = np.cos(2 * np.pi * d.dt.month / 12)
    return df

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def rmsle(y_true, y_pred):
    y_true = np.maximum(np.array(y_true), 0)
    y_pred = np.maximum(np.array(y_pred), 0)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

def time_split(df, task_name, val_months=2):
    sub      = df[df['TaskType'] == task_name].copy()
    date_col = 'SaleDate' if task_name != 'product_weekly' else 'WeekStart'
    cutoff   = sub[date_col].max() - pd.DateOffset(months=val_months)
    tr_part  = sub[sub[date_col] <= cutoff]
    val_part = sub[sub[date_col] >  cutoff]
    print(f"[{task_name}] cutoff={cutoff.date()} | train={len(tr_part)} val={len(val_part)}")
    return tr_part.copy(), val_part.copy(), date_col

print("Helpers defined.")

Helpers defined.


In [8]:
# CELL 4 — Hierarchical features
cat_mean   = train[train['TaskType']=='category_daily'].groupby('Category')['SalesQty'].mean().to_dict()
store_mean = train[train['TaskType']=='store_daily'].groupby('Store')['SalesQty'].mean().to_dict()
prod_mean  = train[train['TaskType']=='product_weekly'].groupby('Product')['SalesQty'].mean().to_dict()

global_mean_log = {
    'store_daily':    np.log1p(np.mean(list(store_mean.values()))) if store_mean else 0.0,
    'category_daily': np.log1p(np.mean(list(cat_mean.values())))   if cat_mean   else 0.0,
    'product_weekly': np.log1p(np.mean(list(prod_mean.values())))  if prod_mean  else 0.0,
}

train['hier_category_mean'] = train['Category'].map(cat_mean)
train['hier_store_mean']    = train['Store'].map(store_mean)
train['hier_product_mean']  = train['Product'].map(prod_mean)
print("Hierarchical features added.")

Hierarchical features added.


In [10]:
# CELL 5 — Core forecast function (ZERO-AWARE FIXED BASELINE)

def run_forecast(task_name, tr, te, mode='val', sparse_threshold=1.0):

    if task_name == 'store_daily':
        group_cols = ['Store'];    date_col = 'SaleDate'
        lags = [1, 7, 14, 28];     roll_wins = [7, 14, 28]
        cat_cols = ['Store'];      hier_cols = ['hier_store_mean']
        use_sparse = False

    elif task_name == 'category_daily':
        group_cols = ['Category']; date_col = 'SaleDate'
        lags = [1, 7, 14, 28];     roll_wins = [7, 14, 28]
        cat_cols = ['Category'];   hier_cols = ['hier_category_mean']
        use_sparse = False

    else:  # product_weekly
        group_cols = ['Product']
        date_col = 'WeekStart'
        lags = [1, 2, 4, 8, 13, 26, 52]
        roll_wins = [2, 4, 8, 13]
        cat_cols = ['Product']
        hier_cols = ['hier_product_mean']
        use_sparse = True

    fb = global_mean_log[task_name]

    tr = tr.copy()
    te = te.copy()
    tr[date_col] = pd.to_datetime(tr[date_col])
    te[date_col] = pd.to_datetime(te[date_col])

    print(f"\n{'='*55}")
    print(f"[{task_name}] mode={mode} | train={len(tr)} | predict={len(te)}")
    print(f"  group_cols={group_cols}")

    if len(tr) == 0 or len(te) == 0:
        return None

    tr = tr.sort_values(group_cols + [date_col]).copy()

    def add_series_key(df):
        df = df.copy()
        if len(group_cols) == 1:
            df['_series_key'] = df[group_cols[0]].map(lambda x: (x,))
        else:
            df['_series_key'] = list(map(tuple, df[group_cols].values))
        return df

    tr = add_series_key(tr)

    sparse_keys = set()
    if use_sparse:
        avg_map = tr.groupby('_series_key')['SalesQty'].mean().to_dict()
        sparse_keys = {k for k, v in avg_map.items() if v < sparse_threshold}
        print(f"  Sparse series (avg < {sparse_threshold}): {len(sparse_keys)} / {len(avg_map)}")

        zero_ratio_map = tr.groupby('_series_key')['SalesQty'].apply(lambda x: (x == 0).mean()).to_dict()
        tr['feat_zero_ratio'] = tr['_series_key'].map(zero_ratio_map).fillna(0)
        tr['feat_is_sparse'] = tr['_series_key'].map(lambda k: int(k in sparse_keys))
    else:
        zero_ratio_map = {}
        tr['feat_zero_ratio'] = 0.0
        tr['feat_is_sparse'] = 0

    tr['target_log'] = np.log1p(tr['SalesQty'])

    # -----------------------------
    # Lag features
    # -----------------------------
    lag_cols = []
    for lag in lags:
        col = f'lag_{lag}'
        tr[col] = tr.groupby('_series_key')['target_log'].shift(lag)
        lag_cols.append(col)

    # -----------------------------
    # Rolling features
    # -----------------------------
    roll_cols = []
    shifted_log = tr.groupby('_series_key')['target_log'].shift(1)

    for w in roll_wins:
        tr[f'roll_mean_{w}'] = shifted_log.groupby(tr['_series_key']).transform(
            lambda x: x.rolling(w, min_periods=1).mean()
        )
        tr[f'roll_std_{w}'] = shifted_log.groupby(tr['_series_key']).transform(
            lambda x: x.rolling(w, min_periods=2).std()
        ).fillna(0)
        tr[f'roll_median_{w}'] = shifted_log.groupby(tr['_series_key']).transform(
            lambda x: x.rolling(w, min_periods=1).median()
        )
        roll_cols += [f'roll_mean_{w}', f'roll_std_{w}', f'roll_median_{w}']

    # -----------------------------
    # Zero-aware features  ← DISABLED (overpredicts on sparse)
    # -----------------------------
    zero_cols = []  # empty — zero_cols removed from feat_cols

    # for lag in lags:
    #     sales_lag_col = f'sales_lag_{lag}'
    #     nz_col = f'lag_{lag}_nonzero'
    #     tr[sales_lag_col] = tr.groupby('_series_key')['SalesQty'].shift(lag)
    #     tr[nz_col] = (tr[sales_lag_col] > 0).astype(int)
    #     zero_cols.append(nz_col)

    # shifted_sales = tr.groupby('_series_key')['SalesQty'].shift(1)
    # for w in roll_wins:
    #     col = f'roll_nonzero_rate_{w}'
    #     tr[col] = shifted_sales.groupby(tr['_series_key']).transform(
    #         lambda x: (x > 0).rolling(w, min_periods=1).mean()
    #     )
    #     zero_cols.append(col)

    # def weeks_since_last_sale(x):
    #     out = []
    #     last = None
    #     for i, v in enumerate(x):
    #         if v > 0:
    #             last = i
    #             out.append(0)
    #         else:
    #             out.append(999 if last is None else i - last)
    #     return pd.Series(out, index=x.index)
    # tr['weeks_since_sale'] = tr.groupby('_series_key')['SalesQty'].transform(weeks_since_last_sale)
    # zero_cols.append('weeks_since_sale')

    # -----------------------------
    # Lag summaries
    # -----------------------------
    tr['lag_mean_short'] = tr[[c for c in ['lag_1', 'lag_2', 'lag_4'] if c in tr.columns]].mean(axis=1)
    tr['lag_mean_long'] = tr[[c for c in ['lag_8', 'lag_13', 'lag_26', 'lag_52'] if c in tr.columns]].mean(axis=1)
    tr['lag_max'] = tr[lag_cols].max(axis=1)
    tr['lag_min'] = tr[lag_cols].min(axis=1)

    feat_cols = (
        lag_cols + roll_cols +
        ['lag_mean_short', 'lag_mean_long', 'lag_max', 'lag_min',
         'feat_zero_ratio', 'feat_is_sparse']
    )

    tr = add_date_features(tr, date_col)

    group_mean_map = tr.groupby('_series_key')['SalesQty'].mean().to_dict()
    tr['hier_group_mean'] = tr['_series_key'].map(group_mean_map)

    date_feats = [c for c in tr.columns if c.startswith('feat_')]
    all_feats = cat_cols + date_feats + feat_cols + hier_cols + ['hier_group_mean']
    all_feats = list(dict.fromkeys([c for c in all_feats if c in tr.columns]))

    if task_name == 'product_weekly':
        core_lags = [c for c in ['lag_1', 'lag_2', 'lag_4'] if c in tr.columns]
    else:
        core_lags = [c for c in ['lag_1', 'lag_7', 'lag_14'] if c in tr.columns]

    tr_clean = tr.dropna(subset=core_lags).copy()

    fill_cols = lag_cols + roll_cols + [
        'lag_mean_short', 'lag_mean_long', 'lag_max', 'lag_min'
    ]

    for col in fill_cols:
        if col in tr_clean.columns:
            tr_clean[col] = tr_clean[col].fillna(0)

    for col in ['hier_group_mean'] + hier_cols:
        if col in tr_clean.columns:
            tr_clean[col] = tr_clean[col].fillna(tr_clean[col].median())

    print(f"  After dropna: {len(tr_clean)} rows")

    if mode == 'val' and len(tr_clean) > 0:
        naive_pred = np.maximum(np.expm1(tr_clean['lag_1']), 0)
        naive_s = smape(tr_clean['SalesQty'], naive_pred)
        print(f"  Train sanity lag_1 SMAPE: {naive_s:.3f}%")

    X_train = tr_clean[all_feats].copy()
    y_train = tr_clean['target_log'].values

    for col in cat_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].astype('category')

    model = LGBMRegressor(
        objective='regression_l1',
        n_estimators=1200,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.2,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        X_train,
        y_train,
        categorical_feature=[c for c in cat_cols if c in X_train.columns]
    )

    history = (
        tr.sort_values(group_cols + [date_col])
          .groupby('_series_key')['target_log']
          .apply(list)
          .to_dict()
    )

    hier_product_map = (
        tr.groupby('_series_key')[hier_cols[0]].median().to_dict()
        if hier_cols and hier_cols[0] in tr.columns else {}
    )
    group_mean_map_log = {k: np.log1p(v) for k, v in group_mean_map.items() if pd.notna(v)}

    def safe_lag(k, l):
        vals = history.get(k, [])
        if len(vals) >= l:
            return vals[-l]
        if len(vals) > 0:
            return vals[-1]
        if k in group_mean_map_log:
            return group_mean_map_log[k]
        hv = hier_product_map.get(k)
        if hv is not None and pd.notna(hv):
            return np.log1p(hv)
        return fb

    def get_weeks_since_sale(k):
        vals = np.expm1(history.get(k, []))
        if len(vals) == 0:
            return 999
        nz = np.where(vals > 0)[0]
        if len(nz) == 0:
            return 999
        return len(vals) - 1 - nz[-1]

    te_s = te.sort_values(group_cols + [date_col]).copy()
    te_s = add_series_key(te_s)
    te_s = add_date_features(te_s, date_col)

    if use_sparse:
        te_s['feat_zero_ratio'] = te_s['_series_key'].map(zero_ratio_map).fillna(0)
        te_s['feat_is_sparse'] = te_s['_series_key'].map(lambda k: int(k in sparse_keys)).astype(int)
    else:
        te_s['feat_zero_ratio'] = 0.0
        te_s['feat_is_sparse'] = 0

    te_s['hier_group_mean'] = te_s['_series_key'].map(group_mean_map)

    for col in hier_cols:
        if col in te_s.columns:
            te_s[col] = te_s[col].fillna(tr[col].median() if col in tr.columns else 0)

    te_s['hier_group_mean'] = te_s['hier_group_mean'].fillna(tr['hier_group_mean'].median())

    preds_all, ids_all, true_all, keys_all = [], [], [], []

    for step_date in sorted(te_s[date_col].dropna().unique()):
        rows = te_s[te_s[date_col] == step_date].copy()

        for lag in lags:
            rows[f'lag_{lag}'] = rows['_series_key'].map(lambda k, l=lag: safe_lag(k, l))
            rows[f'sales_lag_{lag}'] = np.maximum(np.expm1(rows[f'lag_{lag}']), 0)
            rows[f'lag_{lag}_nonzero'] = (rows[f'sales_lag_{lag}'] > 0).astype(int)

        for w in roll_wins:
            rows[f'roll_mean_{w}'] = rows['_series_key'].map(
                lambda k, ww=w: np.mean(history.get(k, [fb])[-ww:])
            )
            rows[f'roll_std_{w}'] = rows['_series_key'].map(
                lambda k, ww=w: np.std(history.get(k, [fb])[-ww:])
                if len(history.get(k, [])) > 1 else 0.0
            )
            rows[f'roll_median_{w}'] = rows['_series_key'].map(
                lambda k, ww=w: np.median(history.get(k, [fb])[-ww:])
            )
            rows[f'roll_nonzero_rate_{w}'] = rows['_series_key'].map(
                lambda k, ww=w: np.mean(np.expm1(history.get(k, [fb])[-ww:]) > 0)
            )

        rows['weeks_since_sale'] = rows['_series_key'].map(get_weeks_since_sale)

        rows['lag_mean_short'] = rows[[c for c in ['lag_1', 'lag_2', 'lag_4'] if c in rows.columns]].mean(axis=1)
        rows['lag_mean_long'] = rows[[c for c in ['lag_8', 'lag_13', 'lag_26', 'lag_52'] if c in rows.columns]].mean(axis=1)
        rows['lag_max'] = rows[lag_cols].max(axis=1)
        rows['lag_min'] = rows[lag_cols].min(axis=1)

        for col in all_feats:
            if col not in rows.columns:
                rows[col] = 0

        X_step = rows[all_feats].copy()

        for col in cat_cols:
            if col in X_step.columns:
                X_step[col] = X_step[col].astype('category')

        log_pred = model.predict(X_step)
        pred = np.maximum(np.expm1(log_pred), 0)

        # Sparse postprocess: threshold 1.0 (rollback from 0.25)
        if use_sparse:
            is_sp = rows['_series_key'].map(lambda k: k in sparse_keys).values
            pred = np.where(is_sp & (pred < 1.0), 0.0, pred)

        for i, (_, row) in enumerate(rows.iterrows()):
            history.setdefault(row['_series_key'], []).append(np.log1p(pred[i]))

        preds_all.extend(pred)
        keys_all.extend(rows['_series_key'].tolist())

        if 'id' in rows.columns:
            ids_all.extend(rows['id'].tolist())

        if mode == 'val' and 'SalesQty' in rows.columns:
            true_all.extend(rows['SalesQty'].tolist())

    if mode == 'val':
        s = smape(true_all, preds_all)
        r = rmse(true_all, preds_all)
        l = rmsle(true_all, preds_all)

        if use_sparse and sparse_keys:
            sp_flags = [int(k in sparse_keys) for k in keys_all]
            t_sp = [t for t, f in zip(true_all, sp_flags) if f == 1]
            p_sp = [p for p, f in zip(preds_all, sp_flags) if f == 1]
            t_nsp = [t for t, f in zip(true_all, sp_flags) if f == 0]
            p_nsp = [p for p, f in zip(preds_all, sp_flags) if f == 0]

            if t_sp:
                print(f"  Sparse  SMAPE: {smape(t_sp, p_sp):.3f}%  RMSLE: {rmsle(t_sp, p_sp):.5f}  (n={len(t_sp)})")
            if t_nsp:
                print(f"  Normal  SMAPE: {smape(t_nsp, p_nsp):.3f}%  RMSLE: {rmsle(t_nsp, p_nsp):.5f}  (n={len(t_nsp)})")

        print(f"  Pred mean/median/max: {np.mean(preds_all):.3f} / {np.median(preds_all):.3f} / {np.max(preds_all):.3f}")
        print(f"  True mean/median/max: {np.mean(true_all):.3f} / {np.median(true_all):.3f} / {np.max(true_all):.3f}")
        print(f"  ✅ SMAPE: {s:.4f}%  |  RMSLE: {l:.5f}  |  RMSE: {r:.4f}")

        return {'task': task_name, 'smape': s, 'rmsle': l, 'rmse': r, 'n': len(preds_all)}

    return pd.DataFrame({'id': ids_all, 'SalesQty': preds_all})

In [11]:
# CELL 5b — Product Weekly Simple v6 (Learned Week Seasonality + Sparse Logic)
# Параметр: window, sparse_threshold, zero_threshold, use_logmean, cap, use_learned_seasonality


def _resolve_cap(tr, cap):
    """cap='p99' / 'p995' / numeric / None дэмжинэ."""
    if cap is None:
        return None
    if isinstance(cap, str):
        c = cap.lower()
        if c == 'p99':
            return float(np.percentile(tr['SalesQty'].clip(lower=0), 99))
        if c == 'p995':
            return float(np.percentile(tr['SalesQty'].clip(lower=0), 99.5))
    return float(cap)


def learn_week_multiplier(tr, target='SalesQty', clip_low=0.75, clip_high=1.35, smooth=10):
    """
    Train-only week-of-year seasonal multiplier сурна.
    Leakage-free: validation/test date-ийн target ашиглахгүй, зөвхөн tr датагаас сурна.
    smooth өндөр байх тусам multiplier 1.0 рүү ойртож overfit багасна.
    """
    tmp = tr[['WeekStart', target]].copy()
    tmp['WeekStart'] = pd.to_datetime(tmp['WeekStart'])
    tmp['week'] = tmp['WeekStart'].dt.isocalendar().week.astype(int).clip(1, 52)

    global_mean = tmp[target].clip(lower=0).mean()
    stats = tmp.groupby('week')[target].agg(['mean', 'count'])

    # Smoothed multiplier: жижиг sample-тэй week-үүдийг global mean рүү татна
    smoothed_mean = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
    mult = smoothed_mean / (global_mean + 1e-9)
    mult = mult.clip(clip_low, clip_high)

    return mult.to_dict()


def run_product_weekly_simple(
    tr, te, mode='val',
    sparse_threshold=1.0,
    window=4,
    zero_threshold=1.0,
    use_logmean=True,
    cap='p99',
    use_learned_seasonality=True,
    season_clip=(0.75, 1.35),
    season_smooth=10
):
    tr = tr.copy()
    te = te.copy()

    tr['WeekStart'] = pd.to_datetime(tr['WeekStart'])
    te['WeekStart'] = pd.to_datetime(te['WeekStart'])

    tr = tr.sort_values(['Product', 'WeekStart'])

    cap_value = _resolve_cap(tr, cap)

    avg_map         = tr.groupby('Product')['SalesQty'].mean().to_dict()
    sparse_products = {p for p, v in avg_map.items() if v < sparse_threshold}

    # Train-аас week seasonal multiplier сурна
    if use_learned_seasonality:
        week_mult_map = learn_week_multiplier(
            tr,
            clip_low=season_clip[0],
            clip_high=season_clip[1],
            smooth=season_smooth
        )
    else:
        week_mult_map = {}

    history = tr.groupby('Product')['SalesQty'].apply(list).to_dict()

    def make_pred(hist, product, pred_date):
        if len(hist) == 0:
            pred = avg_map.get(product, 0)
        else:
            vals = np.array(hist[-window:])
            if use_logmean:
                pred = np.expm1(np.mean(np.log1p(np.maximum(vals, 0))))
            else:
                pred = np.mean(vals)

        # Sparse product: сүүлийн window бүгд zero бөгөөд base pred бага бол zero болгоно
        if product in sparse_products and pred < zero_threshold:
            recent = hist[-window:] if len(hist) else []
            if len(recent) == 0 or np.sum(np.maximum(recent, 0)) == 0:
                pred = 0.0

        # Learned week-of-year seasonal multiplier
        if use_learned_seasonality and pred > 0:
            week = pd.Timestamp(pred_date).isocalendar().week
            week = int(min(max(week, 1), 52))
            pred *= week_mult_map.get(week, 1.0)

        pred = max(pred, 0)
        if cap_value is not None:
            pred = min(pred, cap_value)
        return pred

    preds_all, ids_all, true_all = [], [], []
    te_s = te.sort_values(['WeekStart', 'Product']).copy()

    for step_date in sorted(te_s['WeekStart'].dropna().unique()):
        rows = te_s[te_s['WeekStart'] == step_date].copy()

        for _, row in rows.iterrows():
            product = row['Product']
            hist    = history.get(product, [])
            pred    = make_pred(hist, product, step_date)

            history.setdefault(product, []).append(pred)
            preds_all.append(pred)

            if 'id' in row.index:
                ids_all.append(row['id'])
            if mode == 'val' and 'SalesQty' in row.index:
                true_all.append(row['SalesQty'])

    if mode == 'val':
        s = smape(true_all, preds_all)
        l = rmsle(true_all, preds_all)
        r = rmse(true_all, preds_all)

        sp_flags = []
        for step_date in sorted(te_s['WeekStart'].dropna().unique()):
            rows = te_s[te_s['WeekStart'] == step_date]
            sp_flags.extend([int(p in sparse_products) for p in rows['Product']])

        t_sp  = [t for t, f in zip(true_all, sp_flags) if f == 1]
        p_sp  = [p for p, f in zip(preds_all, sp_flags) if f == 1]
        t_nsp = [t for t, f in zip(true_all, sp_flags) if f == 0]
        p_nsp = [p for p, f in zip(preds_all, sp_flags) if f == 0]

        print(f"\n{'='*60}")
        print(f"[PW SIMPLE v6] w={window}, sparse={sparse_threshold}, zero={zero_threshold}, "
              f"logmean={use_logmean}, cap={cap} -> {cap_value if cap_value is None else round(cap_value, 3)}")
        print(f"  Learned seasonality: {use_learned_seasonality}, clip={season_clip}, smooth={season_smooth}")
        if use_learned_seasonality:
            sample_mult = {k: round(v, 3) for k, v in list(sorted(week_mult_map.items()))[:5]}
            print(f"  Week multiplier sample: {sample_mult} ...")
        print(f"  Sparse products: {len(sparse_products)} / {len(avg_map)}")
        if t_sp:
            print(f"  Sparse  SMAPE: {smape(t_sp, p_sp):.3f}%  RMSLE: {rmsle(t_sp, p_sp):.5f}  (n={len(t_sp)})")
        if t_nsp:
            print(f"  Normal  SMAPE: {smape(t_nsp, p_nsp):.3f}%  RMSLE: {rmsle(t_nsp, p_nsp):.5f}  (n={len(t_nsp)})")
        print(f"  Pred mean/median/max: {np.mean(preds_all):.3f} / {np.median(preds_all):.3f} / {np.max(preds_all):.3f}")
        print(f"  True mean/median/max: {np.mean(true_all):.3f} / {np.median(true_all):.3f} / {np.max(true_all):.3f}")
        print(f"  ✅ SMAPE: {s:.4f}%  |  RMSLE: {l:.5f}  |  RMSE: {r:.4f}")

        return {'task': 'product_weekly', 'smape': s, 'rmsle': l, 'rmse': r, 'n': len(preds_all)}

    return pd.DataFrame({'id': ids_all, 'SalesQty': preds_all})



In [12]:
# CELL 6 — Local Validation
scores = []
for task in ['store_daily', 'category_daily', 'product_weekly']:
    tr_part, val_part, _ = time_split(train, task, val_months=1)

    if task == 'product_weekly':
        # Best params from grid search: window=6, sparse=0.5, zero=0.5
        score = run_product_weekly_simple(
            tr_part, val_part, mode='val',
            window=6, sparse_threshold=0.5,
            zero_threshold=0.5, use_logmean=True, cap='p99'
        )
    else:
        score = run_forecast(task, tr_part, val_part, mode='val', sparse_threshold=1.0)

    if score:
        scores.append(score)

print("\n" + "="*55)
print("  LOCAL VALIDATION SUMMARY")
print("="*55)
total_n = sum(s['n'] for s in scores)
for s in scores:
    rmsle_val = s.get('rmsle', float('nan'))
    print(f"  {s['task']:<22}  RMSLE={rmsle_val:.5f}   SMAPE={s['smape']:6.3f}%")
w_rmsle = sum(s.get('rmsle', 0)*s['n'] for s in scores) / total_n
w_smape = sum(s['smape']*s['n'] for s in scores) / total_n
print(f"\n  {'WEIGHTED RMSLE':<22}  {w_rmsle:.5f}")
print(f"  {'WEIGHTED SMAPE':<22}  {w_smape:.3f}%")
print("="*55)


[store_daily] cutoff=2025-11-30 | train=6700 val=155

[store_daily] mode=val | train=6700 | predict=155
  group_cols=['Store']
  After dropna: 6630 rows
  Train sanity lag_1 SMAPE: 24.277%
  Pred mean/median/max: 561.282 / 506.016 / 1046.915
  True mean/median/max: 615.556 / 562.100 / 1451.162
  ✅ SMAPE: 17.7256%  |  RMSLE: 0.26600  |  RMSE: 170.9450
[category_daily] cutoff=2025-11-30 | train=8040 val=186

[category_daily] mode=val | train=8040 | predict=186
  group_cols=['Category']
  After dropna: 7956 rows
  Train sanity lag_1 SMAPE: 18.789%
  Pred mean/median/max: 466.933 / 318.055 / 1112.747
  True mean/median/max: 512.963 / 353.000 / 2145.000
  ✅ SMAPE: 15.6404%  |  RMSLE: 0.22819  |  RMSE: 158.8425
[product_weekly] cutoff=2025-11-29 | train=343104 val=8935

[PW SIMPLE v6] w=6, sparse=0.5, zero=0.5, logmean=True, cap=p99 -> 77.0
  Learned seasonality: True, clip=(0.75, 1.35), smooth=10
  Week multiplier sample: {1: 0.761, 2: 0.8, 3: 0.829, 4: 0.899, 5: 0.997} ...
  Sparse product

In [13]:
# CELL 6b — Product Weekly Grid Search
# Хамгийн сайн параметр олохын тулд энэ cell-ийг тусад нь ажиллуул

tr_part, val_part, _ = time_split(train, 'product_weekly', val_months=1)

best_rmsle  = float('inf')
best_params = {}
results_grid = []

for w in [2, 3, 4, 6, 8]:
    for sparse_th in [0.5, 1.0, 1.5, 2.0]:
        for zero_th in [0.5, 1.0, 1.5]:
            res = run_product_weekly_simple(
                tr_part, val_part,
                mode='val',
                window=w,
                sparse_threshold=sparse_th,
                zero_threshold=zero_th,
                use_logmean=True,
                cap='p99'
            )
            entry = {
                'window': w, 'sparse_th': sparse_th,
                'zero_th': zero_th, 'rmsle': res['rmsle'],
                'smape': res['smape']
            }
            results_grid.append(entry)
            if res['rmsle'] < best_rmsle:
                best_rmsle  = res['rmsle']
                best_params = entry.copy()

import pandas as pd
grid_df = pd.DataFrame(results_grid).sort_values('rmsle')
print("\n=== TOP 10 combinations (by RMSLE) ===")
print(grid_df.head(10).to_string(index=False))
print(f"\n🏆 BEST: {best_params}")



[product_weekly] cutoff=2025-11-29 | train=343104 val=8935

[PW SIMPLE v6] w=2, sparse=0.5, zero=0.5, logmean=True, cap=p99 -> 77.0
  Learned seasonality: True, clip=(0.75, 1.35), smooth=10
  Week multiplier sample: {1: 0.761, 2: 0.8, 3: 0.829, 4: 0.899, 5: 0.997} ...
  Sparse products: 308 / 1787
  Sparse  SMAPE: 111.691%  RMSLE: 0.61676  (n=1540)
  Normal  SMAPE: 75.210%  RMSLE: 0.66244  (n=7395)
  Pred mean/median/max: 9.076 / 2.369 / 77.000
  True mean/median/max: 10.678 / 2.000 / 521.000
  ✅ SMAPE: 80.1315%  |  RMSLE: 0.65479  |  RMSE: 18.7616

[PW SIMPLE v6] w=2, sparse=0.5, zero=1.0, logmean=True, cap=p99 -> 77.0
  Learned seasonality: True, clip=(0.75, 1.35), smooth=10
  Week multiplier sample: {1: 0.761, 2: 0.8, 3: 0.829, 4: 0.899, 5: 0.997} ...
  Sparse products: 308 / 1787
  Sparse  SMAPE: 111.691%  RMSLE: 0.61676  (n=1540)
  Normal  SMAPE: 75.210%  RMSLE: 0.66244  (n=7395)
  Pred mean/median/max: 9.076 / 2.369 / 77.000
  True mean/median/max: 10.678 / 2.000 / 521.000
  ✅ SM

In [14]:
# CELL 7 — Submit v7 FINAL (FULL TRAIN + BEST PARAMS)
# Validation-аар сонгосон best config:
#   product_weekly: window=6, sparse_threshold=0.5, zero_threshold=0.5, cap='p99'
# Final submission үед validation-д тасалж байсан хэсгийг салгахгүй — train.csv бүх мөрөөр сургана.
# run_product_weekly_simple дотроо p99 cap + learned week multiplier-ийг tr_full буюу FULL TRAIN дээр дахин сурна.

import os

TEST_PATHS = [
    '/kaggle/input/competitions/ai-legends-2026-model-development/test.csv',
    '/kaggle/input/ai-legends-2026-model-development/test.csv',
    '/mnt/data/test.csv',
    'test.csv'
]

for _p in TEST_PATHS:
    if os.path.exists(_p):
        TEST_PATH = _p
        break
else:
    raise FileNotFoundError('test.csv олдсонгүй. TEST_PATHS жагсаалтад зөв path нэмээрэй.')

print('Using TEST_PATH:', TEST_PATH)

test = pd.read_csv(TEST_PATH, low_memory=False)

test['SaleDate']  = pd.to_datetime(test['SaleDate'],  errors='coerce')
test['WeekStart'] = pd.to_datetime(test['WeekStart'], errors='coerce')

test['hier_category_mean'] = test['Category'].map(cat_mean)
test['hier_store_mean']    = test['Store'].map(store_mean)
test['hier_product_mean']  = test['Product'].map(prod_mean)

# --- FINAL BEST PARAMS ---
PW_BEST = dict(
    window=6,
    sparse_threshold=0.5,
    zero_threshold=0.5,
    use_logmean=True,
    cap='p99',
    use_learned_seasonality=True,
    season_clip=(0.75, 1.35),
    season_smooth=10
)

results = []

for task in ['store_daily', 'category_daily', 'product_weekly']:
    te_task = test[test['TaskType'] == task].copy()
    tr_full = train[train['TaskType'] == task].copy()  # FULL TRAIN: val болгон таслахгүй

    print('\n' + '='*60)
    print(f'{task}: train_full={len(tr_full)} | test={len(te_task)}')

    if task == 'product_weekly':
        pred_df = run_product_weekly_simple(
            tr_full,
            te_task,
            mode='test',
            **PW_BEST
        )
    else:
        pred_df = run_forecast(
            task,
            tr_full,
            te_task,
            mode='test',
            sparse_threshold=1.0
        )

    pred_df = pred_df[['id', 'SalesQty']].copy()

    print(
        f"  {task}: pred shape={pred_df.shape}, "
        f"id range=[{pred_df['id'].min()}, {pred_df['id'].max()}], "
        f"pred mean={pred_df['SalesQty'].mean():.3f}, median={pred_df['SalesQty'].median():.3f}, max={pred_df['SalesQty'].max():.3f}"
    )

    results.append(pred_df)

sub = pd.concat(results, axis=0, ignore_index=True)
sub = sub[['id', 'SalesQty']].copy()
sub = sub.sort_values('id').reset_index(drop=True)

# checks
assert len(sub) == len(test), f"Row mismatch: {len(sub)} != {len(test)}"
assert sub['id'].nunique() == len(test), "Duplicate or missing ids"
assert sub['SalesQty'].isna().sum() == 0, "NaN in SalesQty"
assert (sub['SalesQty'] < 0).sum() == 0, "Negative predictions"
assert list(sub.columns) == ['id', 'SalesQty']

sub.to_csv('submission.csv', index=False)

print('\n✅ submission.csv хадгалагдлаа!')
print('Shape:', sub.shape)
print(sub.head())
print(sub.tail())
print(f"SalesQty mean={sub['SalesQty'].mean():.3f}, median={sub['SalesQty'].median():.3f}, max={sub['SalesQty'].max():.3f}, zeros={(sub['SalesQty']==0).sum()}")
print('PW_BEST:', PW_BEST)


Using TEST_PATH: /kaggle/input/competitions/ai-legends-2026-model-development/test.csv

store_daily: train_full=6855 | test=295

[store_daily] mode=test | train=6855 | predict=295
  group_cols=['Store']
  After dropna: 6785 rows
  store_daily: pred shape=(295, 2), id range=[1, 295], pred mean=623.362, median=635.993, max=946.810

category_daily: train_full=8226 | test=354

[category_daily] mode=test | train=8226 | predict=354
  group_cols=['Category']
  After dropna: 8142 rows
  category_daily: pred shape=(354, 2), id range=[296, 649], pred mean=490.780, median=315.999, max=1119.831

product_weekly: train_full=352039 | test=16083
  product_weekly: pred shape=(16083, 2), id range=[650, 16732], pred mean=6.689, median=1.700, max=78.000

✅ submission.csv хадгалагдлаа!
Shape: (16732, 2)
   id    SalesQty
0   1  715.141414
1   2  672.419617
2   3  695.408057
3   4  651.599466
4   5  592.540216
          id   SalesQty
16727  16728  31.267564
16728  16729  31.450039
16729  16730  33.152775
16